# Run → Question Number Map (Full + Targeted)

**Date:** 2026-02-09  
**Purpose:** Map all workflow runs to question numbers, plus extract a targeted subset  
**Outputs:**
- `Run_Question_Map_YYYY-MM-DD.csv` — all runs
- `Target_Run_Question_Map_YYYY-MM-DD.csv` — specific runs only

In [1]:
import json, re, subprocess, csv
from pathlib import Path
from datetime import date
from typing import Dict, List, Optional

REPO = "D-Enns/metac-bot-template"
WORKFLOW = "dre_run_bot_on_tournament.yaml"
OUTPUT_DIR = Path(r"C:/Users/Donni/projects/metac_bot_Spring_2026/products")
TEST_LIMIT = None  # None = all runs

# Specific runs to extract into the target CSV (empty = skip target CSV)
TARGET_RUNS = []  # e.g. [1237, 1221, 1173, 1164]

print(f"Repo: {REPO}")
print(f"Limit: {TEST_LIMIT or 'all'}")
print(f"Target runs: {len(TARGET_RUNS) or 'none set'}")

Repo: D-Enns/metac-bot-template
Limit: all
Target runs: none set


In [2]:
def strip_ansi_codes(text: str) -> str:
    return re.sub(r'\x1b\[[0-9;]*m', '', text)

def get_workflow_runs(repo: str, workflow: str, limit: Optional[int] = None) -> List[Dict]:
    print("Fetching workflow runs...")
    cmd = ["gh", "api", f"repos/{repo}/actions/workflows/{workflow}/runs", "--paginate"]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
    if result.returncode != 0:
        print(f"Failed: {result.stderr}")
        return []
    clean = strip_ansi_codes(result.stdout)

    # Parse concatenated JSON objects via brace counting
    json_objects = []
    current = ""
    count = 0
    for char in clean:
        current += char
        if char == '{':
            count += 1
        elif char == '}':
            count -= 1
            if count == 0 and current.strip():
                json_objects.append(current.strip())
                current = ""

    runs = []
    for obj_str in json_objects:
        data = json.loads(obj_str)
        if 'workflow_runs' in data:
            runs.extend(data['workflow_runs'])

    simplified = [{'id': r['id'], 'run_number': r['run_number']} for r in runs]
    simplified.sort(key=lambda x: x['run_number'], reverse=True)
    if limit:
        simplified = simplified[:limit]
    print(f"{len(simplified)} runs")
    return simplified

def download_run_log(repo: str, run_id: int) -> Optional[str]:
    cmd = ["gh", "run", "view", str(run_id), "--repo", repo, "--log"]
    try:
        result = subprocess.run(cmd, capture_output=True, timeout=120)
        if result.returncode != 0:
            return None
        try:
            log_text = result.stdout.decode('utf-8', errors='replace')
        except:
            log_text = result.stdout.decode('latin-1', errors='replace')
        return strip_ansi_codes(log_text)
    except Exception as e:
        print(f" Error: {e}")
        return None

print("Functions defined")

Functions defined


In [3]:
QUESTION_URL = re.compile(r'https://www\.metaculus\.com/questions/(\d+)')
FOUND_RESEARCH = re.compile(r'Found Research for URL')

runs = get_workflow_runs(REPO, WORKFLOW, limit=TEST_LIMIT)
results = []

for i, run in enumerate(runs, 1):
    rn = run['run_number']
    print(f"[{i}/{len(runs)}] Run #{rn}...", end='')
    log = download_run_log(REPO, run['id'])
    if not log:
        print(" failed")
        results.append((rn, ''))
        continue
    qnum = ''
    if FOUND_RESEARCH.search(log):
        m = QUESTION_URL.search(log)
        if m:
            qnum = m.group(1)
    results.append((rn, qnum))
    print(f" Q:{qnum}" if qnum else " -")

with_q = sum(1 for _, q in results if q)
print(f"\nDone: {len(results)} runs, {with_q} with questions")

Fetching workflow runs...
1260 runs
 Q:42046 Run #1260...
 -/1260] Run #1259...
 -/1260] Run #1258...
 -/1260] Run #1257...
 Q:42045 Run #1256...
 -/1260] Run #1255...
 Q:42044 Run #1254...
 Q:42043 Run #1253...
 Q:42042 Run #1252...
 -0/1260] Run #1251...
 -1/1260] Run #1250...
 -2/1260] Run #1249...
 Q:42078] Run #1248...
 -4/1260] Run #1247...
 -5/1260] Run #1246...
 -6/1260] Run #1245...
 -7/1260] Run #1244...
 -8/1260] Run #1243...
 -9/1260] Run #1242...
 -0/1260] Run #1241...
 -1/1260] Run #1240...
 -2/1260] Run #1239...
 -3/1260] Run #1238...
 Q:42040] Run #1237...
 -5/1260] Run #1236...
 -6/1260] Run #1235...
 -7/1260] Run #1234...
 -8/1260] Run #1233...
 -9/1260] Run #1232...
 -0/1260] Run #1231...
 Q:42039] Run #1230...
 -2/1260] Run #1229...
 Q:42077] Run #1228...
 -4/1260] Run #1227...
 -5/1260] Run #1226...
 -6/1260] Run #1225...
 -7/1260] Run #1224...
 -8/1260] Run #1223...
 -9/1260] Run #1222...
 Q:42038] Run #1221...
 -1/1260] Run #1220...
 -2/1260] Run #1219...
 -3/126

In [4]:
# Write full CSV
results.sort(key=lambda x: x[0], reverse=True)
full_csv = OUTPUT_DIR / f"Run_Question_Map_{date.today()}.csv"

with open(full_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['workflow_run_number', 'question_number'])
    writer.writerows(results)

print(f"Full CSV: {full_csv.name} ({len(results)} rows)")

# Write target CSV (if TARGET_RUNS is set)
if TARGET_RUNS:
    target_set = set(TARGET_RUNS)
    target_results = [(rn, qn) for rn, qn in results if rn in target_set]
    target_csv = OUTPUT_DIR / f"Target_Run_Question_Map_{date.today()}.csv"

    with open(target_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['workflow_run_number', 'question_number'])
        writer.writerows(target_results)

    print(f"Target CSV: {target_csv.name} ({len(target_results)} rows)")
    missing = target_set - {rn for rn, _ in target_results}
    if missing:
        print(f"Warning: runs not found in data: {sorted(missing)}")
else:
    print("No TARGET_RUNS set, skipping target CSV")

Full CSV: Run_Question_Map_2026-02-10.csv (1260 rows)
No TARGET_RUNS set, skipping target CSV
